In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

In [2]:
leagues = ["bundesliga","la_liga","ligue_1","premier_league","serie_a"]
years = ["2020","2021","2022","2023","2024"]

full_data = []

In [3]:
matches_list = []
teams_list = []
players_list = []
for league in leagues:
    for year in years:
        df = pd.read_json(f"../data/raw/understat/{league}/matches_{year}.json")
        df['league'] = league
        df['year'] = year
        matches_list.append(df)

        df2=pd.read_json(f"../data/raw/understat/{league}/players_{year}.json")
        df2['league'] = league
        df2['year'] = year
        players_list.append(df2)

        with open(f"../data/raw/understat/{league}/teams_{year}.json") as f:
            raw = json.load(f)
        for team_id,team_data in raw.items():
            df1 = pd.DataFrame(team_data['history'])
            df1['team_id'] = team_id
            df1['team_name'] = team_data['title']
            df1['league'] = league
            df1['year'] = year
            teams_list.append(df1)

matches_df = pd.concat(matches_list, ignore_index = True)
teams_df = pd.concat(teams_list, ignore_index = True)
players_df = pd.concat(players_list, ignore_index = True)

print("matches_df: ", matches_df.shape)
print("teams_df: ", teams_df.shape)
print("players_df: ", players_df.shape)

matches_df:  (8982, 10)
teams_df:  (17964, 23)
players_df:  (13964, 20)


In [4]:
matches_df['match_id']      = matches_df['id']
matches_df['home_team_id']  = matches_df['h'].apply(lambda x: x['id'])
matches_df['home_team']     = matches_df['h'].apply(lambda x: x['title'])
matches_df['home_short']    = matches_df['h'].apply(lambda x: x['short_title'])
matches_df['away_team_id']  = matches_df['a'].apply(lambda x: x['id'])
matches_df['away_team']     = matches_df['a'].apply(lambda x: x['title'])
matches_df['away_short']    = matches_df['a'].apply(lambda x: x['short_title'])
matches_df['home_goals']    = matches_df['goals'].apply(lambda x: int(x['h']))
matches_df['away_goals']    = matches_df['goals'].apply(lambda x: int(x['a']))
matches_df['home_xG']       = matches_df['xG'].apply(lambda x: float(x['h']))
matches_df['away_xG']       = matches_df['xG'].apply(lambda x: float(x['a']))

matches_df = matches_df.drop(columns=['id', 'h', 'a', 'goals', 'xG', 'forecast', 'isResult'])

print(matches_df.columns.tolist())
print(matches_df.shape)

['datetime', 'league', 'year', 'match_id', 'home_team_id', 'home_team', 'home_short', 'away_team_id', 'away_team', 'away_short', 'home_goals', 'away_goals', 'home_xG', 'away_xG']
(8982, 14)


In [5]:
matches_df.head()
matches_df.dtypes

datetime        datetime64[us]
league                     str
year                       str
match_id                 int64
home_team_id               str
home_team                  str
home_short                 str
away_team_id               str
away_team                  str
away_short                 str
home_goals               int64
away_goals               int64
home_xG                float64
away_xG                float64
dtype: object

In [6]:
teams_df.head()
teams_df.dtypes

h_a                 str
xG              float64
xGA             float64
npxG            float64
npxGA           float64
ppda             object
ppda_allowed     object
deep              int64
deep_allowed      int64
scored            int64
missed            int64
xpts            float64
result              str
date                str
wins              int64
draws             int64
loses             int64
pts               int64
npxGD           float64
team_id             str
team_name           str
league              str
year                str
dtype: object

In [7]:
players_df.head()
players_df.dtypes

id                int64
player_name         str
games             int64
time              int64
goals             int64
xG              float64
assists           int64
xA              float64
shots             int64
key_passes        int64
yellow_cards      int64
red_cards         int64
position            str
team_title          str
npg               int64
npxG            float64
xGChain         float64
xGBuildup       float64
league              str
year                str
dtype: object

In [8]:
matches_df.head()

,datetime,league,year,match_id,home_team_id,home_team,home_short,away_team_id,away_team,away_short,home_goals,away_goals,home_xG,away_xG
0,2020-09-18 18:30:00,bundesliga,2020,14173,117,Bayern Munich,BAY,124,Schalke 04,SCH,8,0,4.620880,0.149750
1,2020-09-19 13:30:00,bundesliga,2020,14174,132,Eintracht Frankfurt,EIN,262,Arminia Bielefeld,ARM,1,1,2.469280,0.618787
2,2020-09-19 13:30:00,bundesliga,2020,14175,240,Union Berlin,UNI,121,Augsburg,AUG,1,3,1.045150,1.420460
3,2020-09-19 13:30:00,bundesliga,2020,14176,134,FC Cologne,COL,120,Hoffenheim,HOF,2,3,2.408010,2.903200
4,2020-09-19 13:30:00,bundesliga,2020,14177,123,Werder Bremen,WER,122,Hertha Berlin,HER,1,4,0.495892,2.054060


In [9]:
teams_df.head()

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,date,wins,draws,loses,pts,npxGD,team_id,team_name,league,year
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,2020-09-18 18:30:00,1,0,0,3,3.713360,117,Bayern Munich,bundesliga,2020
1,a,1.17311,2.523390,1.17311,1.765620,"{'att': 99, 'def': 17}","{'att': 341, 'def': 26}",11,6,1,...,2020-09-27 13:30:00,0,0,1,0,-0.592510,117,Bayern Munich,bundesliga,2020
2,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,2020-10-04 16:00:00,1,0,0,3,1.990240,117,Bayern Munich,bundesliga,2020
3,a,3.29447,1.212350,3.29447,1.212350,"{'att': 223, 'def': 30}","{'att': 416, 'def': 12}",16,4,4,...,2020-10-17 16:30:00,1,0,0,3,2.082120,117,Bayern Munich,bundesliga,2020
4,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,2020-10-24 13:30:00,1,0,0,3,3.379001,117,Bayern Munich,bundesliga,2020


In [10]:
matches_df['league'].value_counts()

league
la_liga           1900
premier_league    1900
serie_a           1900
ligue_1           1752
bundesliga        1530
Name: count, dtype: int64

In [11]:
teams_df.head()

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,date,wins,draws,loses,pts,npxGD,team_id,team_name,league,year
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,2020-09-18 18:30:00,1,0,0,3,3.713360,117,Bayern Munich,bundesliga,2020
1,a,1.17311,2.523390,1.17311,1.765620,"{'att': 99, 'def': 17}","{'att': 341, 'def': 26}",11,6,1,...,2020-09-27 13:30:00,0,0,1,0,-0.592510,117,Bayern Munich,bundesliga,2020
2,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,2020-10-04 16:00:00,1,0,0,3,1.990240,117,Bayern Munich,bundesliga,2020
3,a,3.29447,1.212350,3.29447,1.212350,"{'att': 223, 'def': 30}","{'att': 416, 'def': 12}",16,4,4,...,2020-10-17 16:30:00,1,0,0,3,2.082120,117,Bayern Munich,bundesliga,2020
4,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,2020-10-24 13:30:00,1,0,0,3,3.379001,117,Bayern Munich,bundesliga,2020


In [12]:
players_df.head()

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup,league,year
0,227,Robert Lewandowski,29,2467,41,32.077339,7,4.815550,135,32,4,0,F S,Bayern Munich,33,25.257350,31.740188,5.689343,bundesliga,2020
1,6170,André Silva,32,2787,28,25.599132,5,5.467086,114,31,1,0,F,Eintracht Frankfurt,21,20.294786,26.746589,4.096429,bundesliga,2020
2,8260,Erling Haaland,28,2416,27,23.598484,6,4.035448,92,27,2,0,F S,Borussia Dortmund,25,20.567369,27.197287,5.896189,bundesliga,2020
3,956,Andrej Kramaric,28,2386,20,15.525600,5,4.072423,95,38,2,0,F M S,Hoffenheim,15,11.736730,18.057688,5.372287,bundesliga,2020
4,7052,Wout Weghorst,34,2954,20,18.310400,8,5.427310,93,45,3,0,F S,Wolfsburg,18,16.037062,24.294329,5.954853,bundesliga,2020


In [13]:
matches_df.head()

,datetime,league,year,match_id,home_team_id,home_team,home_short,away_team_id,away_team,away_short,home_goals,away_goals,home_xG,away_xG
0,2020-09-18 18:30:00,bundesliga,2020,14173,117,Bayern Munich,BAY,124,Schalke 04,SCH,8,0,4.620880,0.149750
1,2020-09-19 13:30:00,bundesliga,2020,14174,132,Eintracht Frankfurt,EIN,262,Arminia Bielefeld,ARM,1,1,2.469280,0.618787
2,2020-09-19 13:30:00,bundesliga,2020,14175,240,Union Berlin,UNI,121,Augsburg,AUG,1,3,1.045150,1.420460
3,2020-09-19 13:30:00,bundesliga,2020,14176,134,FC Cologne,COL,120,Hoffenheim,HOF,2,3,2.408010,2.903200
4,2020-09-19 13:30:00,bundesliga,2020,14177,123,Werder Bremen,WER,122,Hertha Berlin,HER,1,4,0.495892,2.054060


In [14]:
matches_df.head()

,datetime,league,year,match_id,home_team_id,home_team,home_short,away_team_id,away_team,away_short,home_goals,away_goals,home_xG,away_xG
0,2020-09-18 18:30:00,bundesliga,2020,14173,117,Bayern Munich,BAY,124,Schalke 04,SCH,8,0,4.620880,0.149750
1,2020-09-19 13:30:00,bundesliga,2020,14174,132,Eintracht Frankfurt,EIN,262,Arminia Bielefeld,ARM,1,1,2.469280,0.618787
2,2020-09-19 13:30:00,bundesliga,2020,14175,240,Union Berlin,UNI,121,Augsburg,AUG,1,3,1.045150,1.420460
3,2020-09-19 13:30:00,bundesliga,2020,14176,134,FC Cologne,COL,120,Hoffenheim,HOF,2,3,2.408010,2.903200
4,2020-09-19 13:30:00,bundesliga,2020,14177,123,Werder Bremen,WER,122,Hertha Berlin,HER,1,4,0.495892,2.054060


In [15]:
teams_df.head()

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,date,wins,draws,loses,pts,npxGD,team_id,team_name,league,year
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,2020-09-18 18:30:00,1,0,0,3,3.713360,117,Bayern Munich,bundesliga,2020
1,a,1.17311,2.523390,1.17311,1.765620,"{'att': 99, 'def': 17}","{'att': 341, 'def': 26}",11,6,1,...,2020-09-27 13:30:00,0,0,1,0,-0.592510,117,Bayern Munich,bundesliga,2020
2,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,2020-10-04 16:00:00,1,0,0,3,1.990240,117,Bayern Munich,bundesliga,2020
3,a,3.29447,1.212350,3.29447,1.212350,"{'att': 223, 'def': 30}","{'att': 416, 'def': 12}",16,4,4,...,2020-10-17 16:30:00,1,0,0,3,2.082120,117,Bayern Munich,bundesliga,2020
4,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,2020-10-24 13:30:00,1,0,0,3,3.379001,117,Bayern Munich,bundesliga,2020


In [16]:
teams_df.shape

(17964, 23)

In [17]:
# derived columns for matches
matches_df['total_goals'] = matches_df['home_goals'] + matches_df['away_goals']
matches_df['total_xG']    = matches_df['home_xG'] + matches_df['away_xG']

# save all three
import os
os.makedirs("../data/processed", exist_ok=True)

matches_df.to_csv("../data/processed/matches.csv", index=False)
players_df.to_csv("../data/processed/players.csv", index=False)
teams_df.to_csv("../data/processed/teams.csv",     index=False)

print("Saved successfully")
print("matches_df:", matches_df.shape)
print("teams_df:  ", teams_df.shape)
print("players_df:", players_df.shape)

Saved successfully
matches_df: (8982, 16)
teams_df:   (17964, 23)
players_df: (13964, 20)


In [18]:
print(matches_df.columns.tolist())

['datetime', 'league', 'year', 'match_id', 'home_team_id', 'home_team', 'home_short', 'away_team_id', 'away_team', 'away_short', 'home_goals', 'away_goals', 'home_xG', 'away_xG', 'total_goals', 'total_xG']


In [19]:
print(teams_df.shape)
print(teams_df.columns.tolist())

(17964, 23)
['h_a', 'xG', 'xGA', 'npxG', 'npxGA', 'ppda', 'ppda_allowed', 'deep', 'deep_allowed', 'scored', 'missed', 'xpts', 'result', 'date', 'wins', 'draws', 'loses', 'pts', 'npxGD', 'team_id', 'team_name', 'league', 'year']
